# RFM Customer Segmentation — Data Exploration
**Dataset:** Synthetic UK-style retail transactions, 2022–2023
**Goal:** Understand data shape, quality, and distributions before RFM scoring.

| Field | Description |
|---|---|
| InvoiceNo | Unique invoice identifier |
| InvoiceDate | Transaction date |
| CustomerID | Unique customer |
| Country | Customer country |
| StockCode | Product ID |
| Category | Product category |
| Quantity | Units purchased |
| UnitPrice | Price per unit (£) |
| Revenue | Quantity × UnitPrice |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", font_scale=1.1)

ROOT = Path("..")
df = pd.read_csv(ROOT / "data" / "retail_transactions.csv", parse_dates=["InvoiceDate"])
print(f"Shape: {df.shape}")
print(f"Date range: {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}")
df.head()

## 1 · Data Quality

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print()
print("Dtypes:")
print(df.dtypes)
print()
print(f"Unique customers : {df['CustomerID'].nunique():,}")
print(f"Unique invoices  : {df['InvoiceNo'].nunique():,}")
print(f"Unique products  : {df['StockCode'].nunique():,}")
print(f"Countries        : {df['Country'].nunique()}")

In [ ]:
print("Revenue summary:")
print(df["Revenue"].describe().round(2))
print()
print("Top 5 countries:")
print(df.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head())

## 2 · Revenue Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Revenue per transaction
axes[0].hist(df["Revenue"].clip(upper=200), bins=50, color="#2a9d8f", edgecolor="white")
axes[0].set_title("Revenue per Line Item (clipped £200)")
axes[0].set_xlabel("Revenue (£)")

# Monthly revenue
monthly = df.groupby(df["InvoiceDate"].dt.to_period("M"))["Revenue"].sum()
axes[1].bar(monthly.index.astype(str), monthly.values, color="#264653")
axes[1].set_title("Monthly Revenue")
axes[1].set_xlabel("Month")
axes[1].tick_params(axis="x", rotation=45)

# Category revenue
cat_rev = df.groupby("Category")["Revenue"].sum().sort_values()
axes[2].barh(cat_rev.index, cat_rev.values, color="#e76f51")
axes[2].set_title("Revenue by Category")
axes[2].set_xlabel("Revenue (£)")

plt.tight_layout()
plt.savefig(ROOT / "images" / "01_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 3 · Customer Purchase Frequency

In [ ]:
invoices_per_customer = df.groupby("CustomerID")["InvoiceNo"].nunique()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(invoices_per_customer.clip(upper=30), bins=30,
             color="#2a9d8f", edgecolor="white")
axes[0].set_title("Invoices per Customer (clipped at 30)")
axes[0].set_xlabel("Number of Invoices")

# CDF
sorted_inv = invoices_per_customer.sort_values()
cdf = np.arange(1, len(sorted_inv)+1) / len(sorted_inv)
axes[1].plot(sorted_inv.values, cdf, color="#e76f51", linewidth=2)
axes[1].axhline(0.8, color="gray", linestyle="--", label="80th percentile")
axes[1].set_title("CDF — Invoices per Customer")
axes[1].set_xlabel("Invoices")
axes[1].set_ylabel("Cumulative %")
axes[1].legend()

plt.tight_layout()
plt.savefig(ROOT / "images" / "02_frequency_dist.png", dpi=150, bbox_inches="tight")
plt.show()

pct80 = invoices_per_customer.quantile(0.8)
pct20 = (invoices_per_customer >= pct80).mean()
print(f"Top 20% of customers (≥{pct80:.0f} invoices) → {pct20*100:.0f}% of the customer base")
top20_rev = df[df["CustomerID"].isin(
    invoices_per_customer[invoices_per_customer >= pct80].index
)]["Revenue"].sum() / df["Revenue"].sum()
print(f"Top 20% customers generate {top20_rev*100:.1f}% of total revenue  (Pareto check)")

## 4 · Key Takeaways
- Revenue follows a **power-law** distribution — a small % of customers generate most revenue
- Monthly revenue shows a clear **Q4 seasonal spike** (Nov–Dec)
- Clear candidate for RFM segmentation: wide variance in recency, frequency, and spend

→ Proceed to **02_rfm_analysis.ipynb**